In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history

TRADING_DAYS_PER_YEAR = 252

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

history.head()


## Sortino ratio background
**Sortino Ratio** is an improved modification of the **Sharp ratio** that penalizes a strategy only for losses, completely ignoring profit as a risk factor.

The denominator in Sharpe ratio is full volatility ($ sigma$):$$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$$Standard deviation considers any deviation from the average to be a risk. If the paper falls by $-10 %$ - $\sigma$ increases (Sharpe falls fairly). But if the paper suddenly takes off by $+50 %, sigma$ also rises sharply, and Sharpe penalizes the portfolio for that jump, artificially underestimating the strategy’s total.

#### Sortino coefficient formula
Instead of the overall standard deviation, here is used Downside Deviation:
$$\text{Sortino} = \frac{R_p - R_f}{\sigma_{\text{downside}}},$$
where:
* $R_p - R_f$ - the same excess portfolio yield over risk-free rate (or $T$ target threshold - Target Return);
* $\sigma_{\text{downside}}$ - volatility calculated only on those days when the acsset has fallen below the $T$ target threshold:
$$\sigma_{\text{downside}} = \sqrt{\frac{1}{N} \sum_{t=1}^{N} \min\left(0, R_t - T\right)^2}$$

If the portfolio has increased on a day of $t$, the difference between $R_t - T > 0$, the function $ min(0, \dots)$ turns that day into zero. Profitable days simply do not contribute to the risk denominator.

In [ ]:
from data.processors import log_returns
from src.portfolio import get_risk_free_rate

log_ret = log_returns(history)
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)
num_assets = log_ret.shape[1]
yr_cov = log_ret.cov() * TRADING_DAYS_PER_YEAR  # type: ignore
optimal_weights = np.ones(10) / 10

daily_rf = risk_free_rate / TRADING_DAYS_PER_YEAR
square_negative_deviations = np.minimum(0, optimal_weights @ log_ret.T - daily_rf) ** 2
exact_max_negative_vol = np.sqrt(np.mean(square_negative_deviations)) * np.sqrt(TRADING_DAYS_PER_YEAR)
exact_max_negative_vol


Find the Sortino ratio

In [ ]:
from src.portfolio import get_risk_free_rate, find_max_sortino

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_sortino, stocks_w) = find_max_sortino(tickers_df=history, rf_base='T_BILLS')

max_sortino_ret = max_sortino.tangency_return
max_sortino_vol = max_sortino.tangency_vol
max_sortino_sortino = max_sortino.max_sortino

print(f"Exact Max Sortino Ratio: {max_sortino.max_sortino:.4f}")
print(f"Exact Sortino Return: {max_sortino_ret:.2%}")
print(f"Exact Sortino Volatility: {max_sortino_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w


Find Sharpe ratio

In [ ]:
from src.portfolio import get_risk_free_rate, find_max_sharpe

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_sharpe, stocks_w) = find_max_sharpe(tickers_df=history, rf_base='T_BILLS', cov_model='LEDOIT_WOLF')

exact_max_ret = max_sharpe.tangency_return
exact_max_vol = max_sharpe.tangency_vol
exact_max_sharpe = max_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {max_sharpe.max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_max_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w


Optimize portfolio for the Efficient Frontier visualization

In [ ]:
from src.portfolio import optimize_portfolio, get_risk_free_rate

optimum_df = optimize_portfolio(tickers_df=history, rf_base='T_BILLS', cov_model='LEDOIT_WOLF')
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)
optimum_df

#### Visualize the data

In [ ]:
from data.processors import log_returns

fig, ax = plt.subplots(figsize=(12, 6))

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()
log_ret = log_returns(history)
expected_returns = log_ret.mean() * TRADING_DAYS_PER_YEAR

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(risk_free_rate + 0.01, risk_free_rate + 0.1),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

# equal weights portfolio
x0 = np.ones(num_assets) / num_assets
def_x = float(np.sqrt(x0.T @ yr_cov @ x0))
def_y = np.dot(x0, expected_returns)
ax.scatter(
    def_x,
    def_y,
    color='darkred',
    alpha=0.7,
)
plt.text(def_x + 0.001, def_y + 0.01, 'Equal weights portfolio')

# S&P 500 portfolio
sp500_df = download_tickers_history(start_date, end_date, ['^GSPC'])
sp500_log_ret = log_returns(sp500_df)

sp500_x = sp500_log_ret.std().iloc[0] * np.sqrt(TRADING_DAYS_PER_YEAR)
sp500_y = sp500_log_ret.mean().iloc[0] * TRADING_DAYS_PER_YEAR
ax.scatter(
    sp500_x,
    sp500_y,
    color='red',
    alpha=0.7,
)
plt.text(sp500_x + 0.001, sp500_y + 0.01, 'S&P 500 portfolio')

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the tangency portfolio point
plt.plot(exact_max_vol, exact_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio',
    xy=(exact_max_vol, exact_max_ret),
    xytext=(exact_max_vol + 0.01, exact_max_ret - 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

# mark the max Sortino point
plt.plot(max_sortino_vol, max_sortino_ret, marker="*", markersize=8, markerfacecolor="lightgreen")
plt.annotate(
    'Max Sortino',
    xy=(max_sortino_vol, max_sortino_ret),
    xytext=(max_sortino_vol + 0.01, max_sortino_ret - 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

plt.title('Efficient Frontier')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
